In [1]:
import os
import numpy as np
import decord

from wm_gym.wm_gym_utils import create_temp_video, Image, Video

In [2]:
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

In [3]:
from wm_gym.vlm_reward_model import OpenAIRewardModel

gpt4_rm = OpenAIRewardModel(
    model="gpt-4o",
    api_key=os.environ["OPENAI_API_KEY"],
    reward_query = (
        "Given these consecutive images from a car racing game, analyze the moving direction and the locations of possible obstacles. And answer the question: do you think the car has any collision with obstacles during the process? "
        "In this game, collisions don’t actually cause any damage — even if the car passes through an obstacle, it still counts as a collision."
    ),
    reward_criteria = (
        "Here is the video description: {} "
        "Return 1 if there is not any collision happened between the car and an obstacle, "
        "Return -1 if you believe a collision has already occurred between the car and an obstacle, regardless of whether there was damage."
        "Your response must only contain one of the following: 1 or -1. Do not include any additional explanation or description."
    ),
)


In [4]:
video_path = "/home/andy/matrix/wm_gym/example_video/hit_tree_example.mp4"
video_reader = decord.VideoReader(video_path)
num_frames = len(video_reader)
all_frames = video_reader.get_batch(list(range(num_frames))).asnumpy()
# Convert to PIL images"
pil_frames = [Image.fromarray(frame) for frame in all_frames]

In [5]:
i = 50
drive_normal_clip = pil_frames[i: i+10]
video_path = create_temp_video(drive_normal_clip)
display(Video(video_path, embed=True))
full_response_pil, short_answer_pil = gpt4_rm.analyze(drive_normal_clip)
print("Full response (PIL):", full_response_pil)
print("Extracted answer (PIL):", short_answer_pil)

Full response (PIL): In the images, the car is moving forward on a dirt path in a car racing game. The environment appears to be an open area with sparse vegetation and no visible obstacles directly in the car's path. The car maintains a consistent position relative to the landscape, suggesting a straight trajectory.

Given the open terrain and lack of visible obstacles directly in front of the car, it seems unlikely that the car would collide with any obstacles during this sequence. The path appears clear, and the car seems to be moving smoothly without any immediate threats of collision.
Extracted answer (PIL): 1


In [6]:
i = 160
crash_tree_clip = pil_frames[i: i+16]
video_path = create_temp_video(crash_tree_clip)
display(Video(video_path, embed=True))
full_response_pil, short_answer_pil = gpt4_rm.analyze(crash_tree_clip, down_sample=4)
print("Full response (PIL):", full_response_pil)
print("Extracted answer (PIL):", short_answer_pil)

Full response (PIL): In the sequence of images, the car is moving forward through a path with trees and bushes on either side. Here's the analysis:

1. **Direction**: The car is moving forward, slightly veering to the right as it progresses through the images.

2. **Obstacles**: The main obstacles are the trees and bushes. In the first image, the car is close to the left side, near the trees. As it moves forward, it appears to be heading towards a more open area.

3. **Collision**: In the first image, the car is very close to the trees on the left. By the second image, the car seems to have moved slightly to the right, but the branches are still close. In the third and fourth images, the car appears to be passing through the branches.

Given this analysis, it seems likely that the car has a collision with the branches of the trees, especially in the third and fourth images, as it appears to be moving through them.
Extracted answer (PIL): -1
